# 메모리 관리

In [40]:
from langchain_core.prompts import ChatPromptTemplate 
from langchain_openai import ChatOpenAI 


llm = ChatOpenAI(
    model="gemma-3-4b-it",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 사용자에게 최선의 금융 조언을 제공합니다"),
        ("placeholder", "{messages}"), 
    ]
)

chain = prompt | llm 

ai_msg = chain.invoke(
    {
        "messages" : [
            ("human" , "저축을 늘리기 위해서 무엇을 할 수 있나요?"), # 첫 질문 
            ("ai" , "저축 목표를 설정하고 매달 자동 이체로 일정 금액을 저축하세요."),
            ("human" , "방금 뭐라고 했나요?")
        ],
    }
)

print(ai_msg.content)


죄송합니다. 제가 너무 간결하게 말씀드렸군요.

제가 드린 조언은 **'목표를 정하고, 행동으로 옮기는 습관을 들이라'**는 것입니다.

좀 더 구체적으로 풀어서 설명드리겠습니다. 저축액을 늘리기 위해 효과적으로 할 수 있는 두 가지 핵심 행동은 다음과 같습니다:

### 1. 구체적인 저축 목표를 설정하세요.
막연히 "많이 모으겠다"가 아니라, **언제까지 얼마를 모을 것인지** 목표를 설정해야 동기 부여가 됩니다.
*   (예시) "1년 안에 여행 자금으로 300만 원 모으기" 또는 "퇴근 후 퇴사 시까지 비상금 500만 원 마련하기"

### 2. '남은 돈 저축'이 아닌, '먼저 저축하고 남은 돈으로 생활하기'를 습관화하세요.
가장 중요한 행동 변화입니다. 월급이 들어오면 지출을 하고 남은 돈으로 저축하려고 하면, 그 '남은 돈' 자체가 없거나 매우 적을 수 있습니다.
*   **가장 먼저, 목표 금액을 계좌 이체하여 저축 통장에 넣어버리세요.** 그리고 그 이후에 생활비를 관리해야 합니다. 이것을 '선 저축, 후 소비'라고 부릅니다.

---

**👉 요약하자면:**
저축을 습관적으로 하기 위해서는 **목표 설정 + 자동 이체**가 필수적이며, 저는 이 두 가지를 먼저 시도해 보시라고 말씀드렸습니다.

혹시 지금 본인의 재정 상태(수입, 지출 등)에 대해 더 자세히 말씀해주시면, 현실적인 저축 계획을 함께 짜 드릴 수 있습니다.


#### 자동 대화 이력 관리 

In [41]:
from langchain_core.prompts import ChatPromptTemplate 
from langchain_core.runnables.history import RunnableWithMessageHistory 
from langchain_community.chat_message_histories import ChatMessageHistory 

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사 입니다. 모든 질문에 최선을 다해 답변하십시오."),
        ("placeholder", "{chat_history}"), 
        ("human", "{input}"),
    ],
)

chat_history=ChatMessageHistory()
chain = prompt | llm  

In [42]:
chain_with_message_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: chat_history, 
    input_message_key = "input",
    history_message_key = "chat_history",
)

In [43]:
chain_with_message_history.invoke(
    {"input":"저축을 눌리기 위해서 무엇을 할 수 있나요?"}, 
    {"configurable" : {"session_id" : "unused"}},
).content

'안녕하세요. 재정 목표를 세우고 실행에 옮기시려는 모습 자체가 이미 성공적인 첫걸음을 내디디신 겁니다. 저축을 늘리는 것은 단순히 돈을 아끼는 행위를 넘어, 우리의 소비 습관과 재정 목표를 점검하고 설계하는 전 과정이라고 할 수 있습니다.\n\n저축을 늘리기 위한 방법은 크게 **\'지출 통제\', \'수입 증대\', \'습관 형성 및 자동화\'** 세 가지 축으로 접근할 수 있습니다. 현재 재정 상태에 맞춰 가장 효과적인 방법을 단계별로 제안해 드리겠습니다.\n\n---\n\n### 🔍 STEP 1: 현 상태 진단하기 (가장 중요!)\n\n무작정 아끼려 하기보다는, 돈이 어디로 가고 있는지 그 흐름을 파악하는 것이 가장 빠르고 정확합니다.\n\n**1. 지출 추적 및 기록 (Tracking)**\n*   가장 먼저 해야 할 일입니다. 최소 1개월 동안은 가계부를 쓰거나 카드 지출 내역을 확인하며 모든 돈의 흐름을 기록하세요. 앱이나 스프레드시트를 활용하면 좋습니다.\n*   **목적:** 내가 \'쓸데없다\'고 생각했던 돈이 사실은 나의 가장 큰 지출 누수처였음을 발견하게 됩니다.\n*   **분류:** 식비, 교통비, 구독료(OTT 등), 취미 생활비 등 항목별로 분류하세요.\n\n**2. 지출의 \'누수처\' 찾기 (Finding Leaks)**\n*   한 달치 기록을 본 후, 가장 비중이 높으면서도 줄일 수 있는 항목(예: 배달 음식 값, 충동구매)을 찾아내세요.\n*   이때 중요한 것은 \'의지와 통제\'가 아니라 \'시스템과 습관\'을 바꾸는 것입니다.\n\n---\n\n### 💰 STEP 2: 지출 통제 및 예산 세우기 (Spending Control)\n\n이제 어디로 돈이 새는지 알았으니, 그 흐름을 원하는 곳으로 돌려놓아야 합니다.\n\n**1. 예산 설정 및 \'선 저축, 후 소비\' 원칙 적용 (Pay Yourself First)**\n*   월급이 들어오면, 가장 먼저 정한 저축 목표액을 별도의 저축 통장으로 자동 이체하세요. 남은

In [44]:
chain_with_message_history.invoke(
    {"input" : "내가 방금 뭐라고 했나요?"}, 
    {"configurable" : {"session_id" : "unused"}}
).content

'네, 방금 제가 드린 말씀은 **"저축액을 늘리기 위한 체계적이고 단계적인 재정 관리 로드맵"**이었습니다.\n\n아마도 방금 제가 드린 답변의 내용이 매우 상세하고 길었기 때문에, 혹시 핵심만 기억하기 어려우셨을 수 있으니, 제가 그 모든 내용을 가장 중요한 액션 플랜 세 가지로 압축하여 다시 말씀드리겠습니다.\n\n---\n\n### 🥇 핵심 요약: 저축을 늘리는 3단계 전략\n\n저축액을 늘리는 것은 단순히 아껴 쓰는 것을 넘어, **\'돈의 흐름 자체를 내가 원하는 대로 설계하는 것\'**이 목표입니다. 이를 위한 과정은 다음과 같이 3단계로 이루어져 있습니다.\n\n#### **1️⃣ STEP 1: 진단하기 (Diagnosis) - 내가 어디에 돈을 쓰는지 파악**\n*   **액션:** 최소 1개월 동안 모든 수입과 지출을 철저히 기록하세요. (가계부 쓰기, 앱 활용)\n*   **목표:** 돈이 어디로 새고 있는지(누수처)를 객관적으로 찾아냅니다. \'내가 쓰지 않았다고 생각한 곳\'이 가장 큰 지출처일 가능성이 높습니다.\n\n#### **2️⃣ STEP 2: 통제하기 (Control) - 지출의 흐름을 바꾸고 막기**\n*   **액션 1. 선 저축, 후 소비 원칙:** 월급이 들어오는 순간, 목표 저축액을 가장 먼저 별도 통장으로 자동 이체하세요. (저축액 = 고정 지출로 간주)\n*   **액션 2. 지출 최적화:** 매달 나가는 구독료, 통신비, 보험 등 고정 지출을 점검하고 다운그레이드하여 자동적으로 비용을 줄이세요.\n*   **액션 3. 습관 바꾸기:** 충동구매를 막고, 외식/배달 대신 \'집밥\'을 정례화하여 생활 습관 자체를 개선하는 것이 가장 큰 효과를 봅니다.\n\n#### **3️⃣ STEP 3: 가속화하기 (Accelerate) - 모으는 속도 높이고 돈 불리기**\n*   **액션 1. 수입 늘리기:** 근본적으로 가장 효과적인 방법은 지출을 줄이는 것 이상으로, 부업이나 자기 계발 등을 통해 \'들어오는 돈 자체를 

#### 대화 이력 요약 및 트리밍 


In [45]:
from langchain_core.messages import trim_messages 
from langchain_core.runnables import RunnablePassthrough 
from operator import itemgetter 

trimmer = trim_messages(strategy="last", max_tokens=2, token_counter=len)

chain_with_trimming=(
    RunnablePassthrough.assign(chat_history=itemgetter("chat_history") | trimmer)
    | prompt 
    | llm 
)

chain_with_trimmed_history = RunnableWithMessageHistory(
    chain_with_trimming, 
    lambda session_id : chat_history, 
    input_messages_key="input", 
    history_messages_key="chat_history",
)

In [46]:
chain_with_trimmed_history.invoke(
    {"input" : "저는 5년 내에 집을 사기 위해서 어떤 재정 계획을 세워야 하나요"}, 
    {"configurable" : {"session_id" : "finance_session_1"}}
)

AIMessage(content='정말 구체적이고 목표가 명확한 질문이십니다. 5년이라는 시간은 꿈을 현실로 만들기에 충분하지만, 동시에 가장 집중적이고 체계적인 노력을 요구하는 시기이기도 합니다.\n\n저는 이 목표를 달성하기 위한 재정 계획을 **\'로드맵 (Roadmap)\'** 형태로 드리겠습니다. 이는 단순히 돈을 모으는 것을 넘어, **수입 구조를 재설계**하고 **리스크를 관리**하며 움직이는 5년간의 여정입니다.\n\n---\n\n## 🏠 5년 내 주택 구매를 위한 재정 로드맵 (The 5-Year Sprint)\n\n이 계획은 크게 **목표 설정 → 자금 확보 단계 → 실행 및 관리**의 3단계로 나눌 수 있습니다.\n\n### 🚀 Phase 0: 목표 설정 및 현실 점검 (The Foundation)\n**가장 먼저 해야 할 일은 \'숫자\'를 뽑아내는 것입니다.** 감정적인 목표가 아니라 냉정한 숫자로 시작해야 합니다.\n\n1.  **최종 구매 가격 설정 (Target Price):**\n    *   어떤 지역, 어떤 평수의 집을 목표로 할지 대략적인 범위를 설정하세요. (예: 수도권 외곽 OO 지역, 90m² 기준)\n    *   이것이 여러분의 **최종 목표 금액**입니다.\n\n2.  **필요 자금 산정 (The Gap Analysis):**\n    *   목표 가격의 최소 **20% 이상**은 계약금 및 중도금을 위해 모으는 것이 안정적입니다.\n    *   여기에 취득세, 중개수수료 등 **부가 비용(Transaction Costs)**을 최소 3~5% 이상 더해야 합니다.\n    *   ➡️ **목표 금액 = (주택 가격의 최소 25%) + 취득세 및 부대 비용**\n\n3.  **현실 자금 상태 진단 (The Baseline):**\n    *   현재 저축할 수 있는 월별 금액은 얼마인가요? (A)\n    *   5년 동안 매달 꾸준히 저축하여 목표 금액(B)에 도달하려면, **매달 얼마를 모아야 하는지** 계산하

In [47]:
chain_with_trimmed_history.invoke(
    {"input" : "방금 제가 뭐라고 했나요? "}, 
    {"configurable" : {"session_id" : "finance_session_1"}}
).content

'죄송합니다. 혹시 제가 너무 자세한 설명을 드려서 혼란스러우셨을까 봐 다시 한번 확인드리겠습니다.\n\n방금 고객님께서 말씀하신 내용은 다음과 같습니다.\n\n**👉 "저는 5년 내에 집을 사기 위해서 어떤 재정 계획을 세워야 하나요?"**\n\n이 질문에 대해 제가 드린 답변은, 단순히 \'저축하라\'는 수준의 조언이 아니라, **5년이라는 목표 시점에 맞춰 움직이는 3단계의 종합적인 \'재정 로드맵\'**이었습니다.\n\n제가 드린 답변의 핵심은 이것입니다:\n\n*   **Phase 0 (목표 설정):** 목표 주택 가격을 정하고, 필요한 자금(계약금 + 부대 비용)이 얼마인지 냉정하게 계산해야 합니다.\n*   **Phase 1 (실행 가속화):** 현재 수입과 목표액을 비교하여 부족한 부분을 채울지, 아니면 불필요한 지출을 막고 공격적으로 저축/투자해야 할지를 결정하는 것입니다.\n*   **핵심 성공 요인:** 단기적인 \'짠테크\'를 넘어, 목표 시점에 따라 자금을 안전하게 보존하면서도 복리 효과를 얻을 수 있는 **\'자금의 역할 분리 및 투자 전략\'**이 필요하다는 점을 강조했습니다.\n\n혹시 제가 드린 답변 중 특정 부분(예: "투자 포트폴리오가 궁금해요" 또는 "매달 얼마씩 모아야 하나요?")에 대해 더 자세한 설명이 필요하시다면, 언제든지 다시 말씀해 주세요. 제가 필요한 부분을 압축하여 핵심만 짚어드리겠습니다.'

#### 대화 요약 활용 

In [48]:
def summarize_messages(chain_input):
    stored_messages = chat_history.messages
    if (len(stored_messages)) == 0:
        return False 
    
    summarization_prompt = ChatPromptTemplate.from_messages(
        [
            # 이전 대화 
            ("placeholder", "{chat_history}"), 
            ("user", "이전 대화를 요약해주세요. 가능한 많은 세부 정보를 포함하세요"),
        ],
    )
    
    summarization_chain = summarization_prompt | llm 
    summary_message = summarization_chain.invoke({"chat_history": stored_messages})
    
    chat_history.clear()
    chat_history.add_message(summary_message)
    return True 

    

In [49]:

# 대화 요약을 처리 체인 
chain_with_summarization = (
    RunnablePassthrough.assign(messages_summarized=summarize_messages)
    | chain_with_message_history
)

print(chain_with_summarization.invoke(
    {"input": "저에게 어떤 재정적 조언을 해주셨나요?"},  
    {"configurable": {"session_id": "finance_session_1"}}  
).content)


고객님께서 궁금해하신 부분을 제가 한 번에 압축적으로 정리하여 다시 말씀드리겠습니다.

제가 드린 조언은 단순히 '돈을 모으는 법'에 머무르지 않고, 고객님의 재정 상태를 **진단(Diagnosis) $\rightarrow$ 목표 설정 및 계획 수립(Planning) $\rightarrow$ 실행 및 가속화(Execution)**의 3단계로 나누어 접근한 **종합적인 재정 로드맵**이었습니다.

아래는 대화의 흐름과 핵심적인 조언을 단계별로 정리한 요약본입니다.

---

### 🧭 재정 컨설팅의 전체 흐름: 단계별 조언 요약

#### Phase 1. 기초 진단 및 재정 건전성 확보 (Foundation & Diagnosis)
(초기에는 '저축액 증대'라는 일반적인 목표에 집중했습니다.)

가장 먼저 해야 할 일은 **돈이 나가는 곳을 정확히 아는 것**입니다. 모든 재정 관리는 '현상 파악'에서 시작합니다.

*   **📊 지출 누수 진단 (Leakage Point Identification):** 최소 1개월간의 철저한 지출 추적을 통해 어디로 돈이 새고 있는지(낭비 요소, 비효율적인 고정 지출)를 찾아냈습니다.
*   **🔒 선 저축, 후 소비 원칙 적용 (Pay Yourself First):** 수입이 발생한 즉시 목표 저축액을 가장 먼저 별도 계좌로 자동 이체하는 시스템을 구축하여, 남은 돈으로 생활한다는 원칙을 확립했습니다.
*   **✂️ 비용 구조 최적화 (Cost Optimization):** 불필요한 구독 서비스 해지, 통신 요금제 다운그레이드 등 고정비를 점검하고 줄이는 방안을 모색했습니다.
*   **🧠 행동 경제학 적용 (Behavioral Finance):** 충동구매를 막기 위한 '48시간 숙려 기간' 설정, 식비 통제를 위한 주간 식단 계획(집밥) 실행 등 심리적 습관 변화를 통한 지출 통제력을 높이는 것을 조언했습니다.

#### Phase 2. 고액 목표 구체화 및 로드맵 수립 (Goal Setting & Planning)
(